In [5]:
import sys

import ray
from skyrl.train.config import SkyRLTrainConfig
from skyrl.train.utils import initialize_ray
from skyrl.train.entrypoints.main_base import BasePPOExp, validate_cfg
from skyrl_gym.envs import register

2026-03-12 02:36:33,361	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [6]:
import re
from typing import Any, Dict

from skyrl_gym.envs.base_text_env import BaseTextEnv, BaseTextEnvStepOutput

class MultiplyEnv(BaseTextEnv):
    def __init__(
        self,
        env_config: Dict[str, Any] = {},
        extras: Dict[str, Any] = {},
    ):
        super().__init__()
        assert "reward_spec" in extras, "reward_spec field is required"
        assert "ground_truth" in extras["reward_spec"], "ground_truth is required in reward_spec field"
        self.ground_truth = extras["reward_spec"]["ground_truth"]
        self.max_turns = extras.get("max_turns", 5)
    def _parse_action(self, action: str) -> str:
      """Extract answer from \\boxed{answer} format"""
      match = re.search(r"\\boxed\{([^}]+)\}", action)
      return match.group(1) if match else None

    def step(self, action: str) -> BaseTextEnvStepOutput:
        self.turns += 1
        answer = self._parse_action(action)
        is_correct = answer is not None and answer.strip() == str(self.ground_truth).strip()
        found_boxed = answer is not None
        # Episode ends if max turns reached or correct answer found
        done = self.turns >= self.max_turns or is_correct
        # Reward structure:
        # - Correct answer: 1.0
        # - Wrong answer in correct format: 0.5  
        # - No boxed answer: 0.0
        if is_correct:
            reward = 1.0
        elif found_boxed:
            reward = 0.5
        else:
            reward = 0.0
        if done:
            return BaseTextEnvStepOutput(
                observations=[],
                reward=reward,
                done=True,
                metadata={"parsed_answer": answer}
            )
        # Give feedback for another attempt
        if answer is not None:
            feedback = f"Your answer '{answer}' is incorrect. Please try again."
        else:
            feedback = "Please provide your answer in the format \\boxed{your_answer}."
        return BaseTextEnvStepOutput(
            observations=[{"role": "user", "content": feedback}],
            reward=0.0,
            done=False,
            metadata={"parsed_answer": answer}
        )



In [7]:
# Instantiate with a test case
env = MultiplyEnv(
    env_config={},
    extras={
        "reward_spec": {"method": "rule", "ground_truth": "42"},
        "max_turns": 3,
    }
)

# Test correct answer
out = env.step(r"\boxed{42}")
assert out["reward"] == 1.0
assert out["done"] == True

# Test wrong answer in correct format
env2 = MultiplyEnv(env_config={}, extras={"reward_spec": {"ground_truth": "42"}, "max_turns": 3})
out2 = env2.step(r"\boxed{41}")
print(out2)

# Test no boxed answer
env3 = MultiplyEnv(env_config={}, extras={"reward_spec": {"ground_truth": "42"}, "max_turns": 3})
out3 = env3.step("The answer is 42")
print(out3)

{'observations': [{'role': 'user', 'content': "Your answer '41' is incorrect. Please try again."}], 'reward': 0.0, 'done': False, 'metadata': {'parsed_answer': '41'}}
{'observations': [{'role': 'user', 'content': 'Please provide your answer in the format \\boxed{your_answer}.'}], 'reward': 0.0, 'done': False, 'metadata': {'parsed_answer': None}}


In [1]:
from src.trainer.hf_dataloader import load_combined_qa

combined_qa = load_combined_qa()

/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
combined_qa

Dataset({
    features: ['question', 'answer', 'context', 'year', 'ticker_or_company_name', 'filing_type', 'data_source'],
    num_rows: 7150
})

In [3]:
combined_qa[0]

{'question': 'What area did NVIDIA initially focus on before expanding to other computationally intensive fields?',
 'answer': 'NVIDIA initially focused on PC graphics.',
 'context': 'Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.',
 'year': '2023',
 'ticker_or_company_name': 'NVDA',
 'filing_type': '10K',
 'data_source': 'virattt/financial-qa-10K'}